# Granite Speech 5.0 日本語ASR検証（2B / 470M TurboCTC / 4.1 2B）

- 環境：IBM TechZone / watsonx.ai Studio Notebook（Tesla V100 16GB × 2、Python 3.12）
- バージョン：torch 2.6.0 / torchaudio 2.6.0+cu126 / transformers 5.21.0
- テストセット：日本語の短文音声 5本（レコーディング1〜5、16kHzモノラルWAV）

## 1. セットアップ

In [1]:
!pip install -U transformers soundfile

import transformers
print("transformers:", transformers.__version__)

transformers: 5.21.0


In [3]:
!pip install torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126
import torchaudio
print("torchaudio:", torchaudio.__version__)

Looking in indexes: https://download.pytorch.org/whl/cu126
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 62.8 MB/s  0:00:00
torchaudio: 2.6.0+cu126


> **ここでカーネルを再起動する。**
>
> transformers は import した時点で torchaudio の有無を記録するため、後から torchaudio を入れただけでは `GraniteSpeechFeatureExtractor requires the torchaudio library` のエラーになる。pip install の再実行は不要。

In [1]:
import torch, torchaudio, transformers
print(torch.__version__, torchaudio.__version__, transformers.__version__)

2.6.0 2.6.0+cu126 5.21.0


## 2. Granite Speech 5.0 2B の取得と動作確認

In [2]:
import os
from huggingface_hub import snapshot_download

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"
LOCAL_DIR = "./hf_home/granite-speech-5.0-2b"
snapshot_download(repo_id=MODEL_ID, local_dir=LOCAL_DIR)
print(os.listdir(LOCAL_DIR))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

['model.safetensors', 'multilingual_sample.wav', 'processor_config.json', 'generation_config.json', 'README.md', 'out_llm.safetensors', 'assets', 'config.json', '.cache', '.gitattributes', 'model.sig', 'tokenizer.json', 'vocab.json', 'preprocessor_config.json', 'merges.txt', 'special_tokens_map.json', 'tokenizer_config.json', 'chat_template.jinja', 'added_tokens.json']


In [3]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(LOCAL_DIR)
tokenizer = processor.tokenizer
model = AutoModelForSpeechSeq2Seq.from_pretrained(LOCAL_DIR, device_map=device, dtype=dtype)
model.eval()
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

VRAM used: 4.63 GB


In [4]:
import time

def transcribe(wav, prompt_text, max_new_tokens=400, anti_loop=False):
    chat = [{"role": "user", "content": "<|audio|>" + prompt_text}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, wav, device=device, return_tensors="pt").to(device)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=False, num_beams=1)
    if anti_loop:  # 前回の日本語向けループ対策
        gen_kwargs.update(repetition_penalty=1.3, no_repeat_ngram_size=5)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    n_in = inputs["input_ids"].shape[-1]
    text = tokenizer.batch_decode(out[:, n_in:], add_special_tokens=False, skip_special_tokens=True)[0]
    return text, time.time() - t0

def load_wav(path):
    wav, sr = torchaudio.load(path, normalize=True)
    assert wav.shape[0] == 1 and sr == 16000, f"16kHzモノラルが必要: ch={wav.shape[0]}, sr={sr}"
    print(f"{path}: {wav.shape[1] / sr:.1f} 秒")
    return wav

In [5]:
sample = load_wav(os.path.join(LOCAL_DIR, "multilingual_sample.wav"))
text, sec = transcribe(sample, "transcribe the speech with proper punctuation and capitalization.")
print(f"[{sec:.1f}s] {text}")

./hf_home/granite-speech-5.0-2b/multilingual_sample.wav: 24.9 秒
[3.9s] For Timothy was a spoiled cat, and he allowed no one to interfere. Everybody waited upon him, moving their chairs even, for he was monarch of the hearth. Dinarzade la nuit suivante appela sa soeur quand il en fut temps. "Si vous ne dormez pas, ma soeur," lui dit-elle, "je vous prie en attendant le jour qui paraîtra bientôt de continuer le conte du pêcheur."


## 3. テスト音声の取得（Cloud Object Storage）

プロジェクトにアップロードしたWAVを、ツールバー `</>` のコードスニペットで生成したクライアントで取得する。取得と保存は同じセルで実行する（別セルに分けるとタイムアウトしやすい）。

In [12]:
import ibm_boto3
from botocore.client import Config

# コードスニペットで生成される内容。APIキーとバケット名は各自の値に置き換える（公開しないこと）
cos_client = ibm_boto3.client(service_name='s3',
    ibm_api_key_id='<YOUR_IBM_CLOUD_API_KEY>',
    ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
    config=Config(signature_version='oauth'),
    endpoint_url='https://s3.direct.us-south.cloud-object-storage.appdomain.cloud')

bucket = '<YOUR_BUCKET_NAME>'

for i in range(1, 6):
    object_key = f'レコーディング{i}.wav'
    body = cos_client.get_object(Bucket=bucket, Key=object_key)['Body']
    with open(f"./{object_key}", "wb") as f:
        f.write(body.read())
    print(f"{object_key} 保存完了")

レコーディング1.wav 保存完了
レコーディング2.wav 保存完了
レコーディング3.wav 保存完了
レコーディング4.wav 保存完了
レコーディング5.wav 保存完了


## 4. Granite Speech 4.1 2B と 5.0 2B の比較

4.1 を2枚目のGPUに載せ、同じ音声・同じ条件で推論する。生成パラメーターは前回（4.1の記事）と同じ指定のみ。

In [14]:
LOCAL_DIR_41 = "./hf_home/granite-speech-4.1-2b"
snapshot_download(repo_id="ibm-granite/granite-speech-4.1-2b", local_dir=LOCAL_DIR_41)

processor_41 = AutoProcessor.from_pretrained(LOCAL_DIR_41)
model_41 = AutoModelForSpeechSeq2Seq.from_pretrained(
    LOCAL_DIR_41, device_map="cuda:1", dtype=torch.float16
)
model_41.eval()

MODELS = {
    "4.1": {"model": model_41, "processor": processor_41, "device": "cuda:1"},
    "5.0": {"model": model,    "processor": processor,    "device": "cuda:0"},
}
for d in ["cuda:0", "cuda:1"]:
    print(d, f"{torch.cuda.memory_allocated(d) / 1e9:.2f} GB")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

cuda:0 4.64 GB
cuda:1 4.63 GB


In [15]:
import base64
from IPython.display import HTML, display

PROMPT = "transcribe the speech with proper punctuation and capitalization."
GEN_KW = dict(max_new_tokens=512, repetition_penalty=1.3, no_repeat_ngram_size=5)  # 前回の4.1と同じ指定のみ

def run(name, wav):
    m = MODELS[name]
    tok = m["processor"].tokenizer
    chat = [{"role": "user", "content": "<|audio|>" + PROMPT}]
    prompt = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = m["processor"](prompt, wav, device=m["device"], return_tensors="pt").to(m["device"])
    t0 = time.time()
    with torch.no_grad():
        out = m["model"].generate(**inputs, **GEN_KW)
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, time.time() - t0

def download_link(name, text):
    b64 = base64.b64encode(text.encode()).decode()
    return f'<a href="data:text/plain;base64,{b64}" download="{name}">📄 {name}</a>'

results = {}
for i in range(1, 6):
    wav = load_wav(f"./レコーディング{i}.wav")
    results[i] = {}
    links = []
    for ver in ["4.1", "5.0"]:
        text, sec = run(ver, wav)
        results[i][ver] = {"text": text, "time": sec}
        print(f"  {ver} [{sec:.1f}s] {text}")
        links.append(download_link(f"レコーディング{i}_granite-speech-{ver}.txt", text))
    print("  →", "同一" if results[i]["4.1"]["text"] == results[i]["5.0"]["text"] else "★差分あり")
    display(HTML(" ".join(links)))
    print()

./レコーディング1.wav: 14.3 秒
  4.1 [2.3s] 現在、医療は社員のリスキリングを促進しており特に生成ａｉやデータ分析のスキル修得が急務となっています。
  5.0 [2.1s] 現在、医療は社員のリスキリングを促進しており特に生成ａｉやデータ分析のスキル修得が急務となっています。
  → 同一



./レコーディング2.wav: 15.4 秒
  4.1 [2.2s] ２０１３年度から完全実施予定のインボイス制度に備え、税理部門は適格請求書の管理システムを刷新しました。
  5.0 [2.1s] ２０１３年度から完全実施予定のインボイス制度に備え、税理部門は適格請求書の管理システムを刷新しました。
  → 同一



./レコーディング3.wav: 15.0 秒
  4.1 [2.6s] 会議の効率化に向けて、チャットｇｐｔを活用した記録生成サービスを試験導入し去年度からコスト削減につながる見込みです。
  5.0 [2.5s] 会議の効率化に向けて、チャットｇｐｔを活用した記録生成サービスを試験導入し去年度からコスト削減につながる見込みです。
  → 同一



./レコーディング4.wav: 20.1 秒
  4.1 [2.8s] エンジニアは生成AIの学習モデルを改善するためコードを最適化したと説明しました一方で社内では行動機関いわゆるcodeforconduitの見直しも進めています。
  5.0 [2.7s] エンジニアは生成ａｉの学習モデルを改善するためコードを最適化したと説明しました一方で社内では行動機関いわゆるcodeofconductの見直し�も進めています。
  → ★差分あり



./レコーディング5.wav: 20.5 秒
  4.1 [3.5s] 新しいマーケティング施策ではクラウドを活用したデータとおきばんを導入しあげてそうの分析精度を高めることを目指しています私、クラウドファンディングのような支援型の手法も検討中です。
  5.0 [3.4s] 新しいマーケティング施策ではクラウドを活用したデータとお気転を導入しあげっと層の分析精度を高めることを目指しています私、クラウドファンディングのような支援型の手法も検討中です。
  → ★差分あり


## 5. Granite Speech 5.0 470M TurboCTC

CTC（エンコーダーのみ）のため、`AutoModelForCTC` で読み込み、プロンプトなしで推論する。

In [16]:
from transformers import AutoModelForCTC

LOCAL_DIR_470 = "./hf_home/granite-speech-5.0-470m-turboctc"
snapshot_download(repo_id="ibm-granite/granite-speech-5.0-470m-turboctc", local_dir=LOCAL_DIR_470)

processor_470 = AutoProcessor.from_pretrained(LOCAL_DIR_470)
model_470 = AutoModelForCTC.from_pretrained(
    LOCAL_DIR_470, device_map="cuda:1", dtype=torch.float32  # 小さいのでfloat32で安全に
)
model_470.eval()
print("dtype:", model_470.dtype, "/ cuda:1", f"{torch.cuda.memory_allocated('cuda:1') / 1e9:.2f} GB")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/550 [00:00<?, ?it/s]

dtype: torch.float32 / cuda:1 6.53 GB


In [17]:
def run_470(wav):
    sr = processor_470.feature_extractor.sampling_rate
    inputs = processor_470([wav[0].numpy()], sampling_rate=sr, device=model_470.device)
    inputs = inputs.to(model_470.device, dtype=model_470.dtype)
    t0 = time.time()
    with torch.no_grad():
        out = model_470.generate(**inputs)
    text = processor_470.batch_decode(out, skip_special_tokens=True)[0]
    return text, time.time() - t0

for i in range(1, 6):
    wav = load_wav(f"./レコーディング{i}.wav")
    text, sec = run_470(wav)
    results[i]["470m"] = {"text": text, "time": sec}
    print(f"  2B   [{results[i]['5.0']['time']:.1f}s] {results[i]['5.0']['text']}")
    print(f"  470M [{sec:.2f}s] {text}")
    name = f"レコーディング{i}_granite-speech-5.0-470m.txt"
    display(HTML(download_link(name, text)))
    print()

./レコーディング1.wav: 14.3 秒
  2B   [2.1s] 現在、医療は社員のリスキリングを促進しており特に生成ａｉやデータ分析のスキル修得が急務となっています。
  470M [0.03s] 現在企業は社員のリスキリングを促進しており特に生成aiやデータ分析のスキル習得が急務となっています。



./レコーディング2.wav: 15.4 秒
  2B   [2.1s] ２０１３年度から完全実施予定のインボイス制度に備え、税理部門は適格請求書の管理システムを刷新しました。
  470M [0.03s] 2013年度から完全実施予定のインボイス制度に備え営部門は適格請求書の管理システムを刷新しました



./レコーディング3.wav: 15.0 秒
  2B   [2.5s] 会議の効率化に向けて、チャットｇｐｔを活用した記録生成サービスを試験導入し去年度からコスト削減につながる見込みです。
  470M [0.03s] 会議の効率化に向けてchatgptを活用した持録生成サービスを試験導入し初年度からコスト削減につながる見込みです。



./レコーディング4.wav: 20.1 秒
  2B   [2.7s] エンジニアは生成ａｉの学習モデルを改善するためコードを最適化したと説明しました一方で社内では行動機関いわゆるcodeofconductの見直し�も進めています。
  470M [0.04s] エンジニアは生成aiの学習モデルを改善するため高度を最適化したと説明しました一方で社内では行動規範いわゆる行動ofコンダクトの見直しも進めています



./レコーディング5.wav: 20.5 秒
  2B   [3.4s] 新しいマーケティング施策ではクラウドを活用したデータとお気転を導入しあげっと層の分析精度を高めることを目指しています私、クラウドファンディングのような支援型の手法も検討中です。
  470M [0.04s] 新しいマーケティング施策ではクラウドを活用したデータ統合基盤を導入しターゲット層の分析精度を高めることを目指していますただしクラウドファンディングのような支援型の手法も検討中です。


## 6. 評価結果（WER）

書き起こしを [watson-stt-comparison](https://github.com/matsuo-iguazu/watson-stt-comparison) の評価パイプラインで正規化・トークン化（MeCab）して算出。

```
python run_full_pipeline.py --samples samples/myTestcase --out output/granite5_20261028 --skip-stt
```

| | 5.0 2B | 5.0 470M TurboCTC |
|---|---|---|
| レコーディング1 | 0.107 | 0.036 |
| レコーディング2 | 0.080 | 0.080 |
| レコーディング3 | 0.172 | 0.069 |
| レコーディング4 | 0.150 | 0.100 |
| レコーディング5 | 0.140 | 0.000 |
| 誤り数の合計（165トークン） | 22 | 9 |

4.1 2B の再推論はレコーディング1〜3で5.0 2Bと同一出力、レコーディング1は5月の結果とも一致（参考）。